# 03: ベクターデータベース (VDB) の作成とデプロイ

このノートブックでは、経済産業省の EC 市場調査 PDF をもとにベクターデータベースを構築し、RAG 用にデプロイします。

## 処理の流れ
1. 環境変数の読み込みと DataRobot クライアントの初期化
2. PDF ドキュメントの準備
3. PDF を ZIP に圧縮して AI Catalog にアップロード
4. VectorDatabase の作成
5. VDB 完了の待機
6. カスタムモデルワークショップへの送信
7. モデルビルドの待機
8. モデルパッケージとして登録
9. デプロイ
10. `.env` に設定する VDB_DEPLOYMENT_ID の出力

## 1. 環境変数の読み込み

In [ ]:
import os
import time
import json
import zipfile
import pathlib
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv("../.env", override=True)

DATAROBOT_API_TOKEN = os.environ["DATAROBOT_API_TOKEN"]
DATAROBOT_ENDPOINT = os.environ.get("DATAROBOT_ENDPOINT", "https://app.datarobot.com/api/v2")

print(f"Endpoint: {DATAROBOT_ENDPOINT}")
print(f"Token:    {DATAROBOT_API_TOKEN[:8]}...")

## 2. DataRobot クライアントの初期化

In [ ]:
import datarobot as dr
import requests

client = dr.Client(
    token=DATAROBOT_API_TOKEN,
    endpoint=DATAROBOT_ENDPOINT,
)
print(f"DataRobot client initialized (v{dr.__version__})")

# REST API 用のヘッダー
API_BASE = DATAROBOT_ENDPOINT
HEADERS = {
    "Authorization": f"Bearer {DATAROBOT_API_TOKEN}",
    "Content-Type": "application/json",
}

## 3. PDF ドキュメントの準備

経済産業省の EC 市場調査 PDF を `../documents/` ディレクトリに配置してください。

### 方法 A: 手動配置
`../documents/` フォルダに PDF ファイルを配置してください。

### 方法 B: ダウンロード (以下のセルを実行)
経産省の EC 市場調査レポートをダウンロードします。

In [ ]:
DOCUMENTS_DIR = pathlib.Path("../documents").resolve()
DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)

# 経産省 EC市場調査 PDF のURL (適宜更新してください)
# 例: 令和5年度 電子商取引に関する市場調査
PDF_URL = "https://www.meti.go.jp/press/2024/09/20240925001/20240925001-1.pdf"
PDF_FILENAME = "ec_market_survey.pdf"
PDF_PATH = DOCUMENTS_DIR / PDF_FILENAME

if PDF_PATH.exists():
    print(f"PDF は既に存在します: {PDF_PATH}")
else:
    print(f"PDF をダウンロード中: {PDF_URL}")
    try:
        resp = requests.get(PDF_URL, timeout=120)
        resp.raise_for_status()
        PDF_PATH.write_bytes(resp.content)
        print(f"ダウンロード完了: {PDF_PATH} ({len(resp.content) / 1024 / 1024:.1f} MB)")
    except Exception as e:
        print(f"ダウンロードに失敗しました: {e}")
        print(f"手動で PDF を {DOCUMENTS_DIR} に配置してください。")

In [ ]:
# documents ディレクトリ内の PDF を確認
pdf_files = list(DOCUMENTS_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(
        f"PDF ファイルが見つかりません。{DOCUMENTS_DIR} にPDFを配置してください。"
    )

print(f"使用する PDF ファイル:")
for f in pdf_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

## 4. PDF を ZIP に圧縮して AI Catalog にアップロード

DataRobot の VectorDatabase はZIPファイルからドキュメントを読み込みます。

In [ ]:
ZIP_PATH = DOCUMENTS_DIR / "ec_documents.zip"

with zipfile.ZipFile(str(ZIP_PATH), "w", zipfile.ZIP_DEFLATED) as zf:
    for pdf_file in pdf_files:
        zf.write(str(pdf_file), pdf_file.name)
        print(f"  ZIP に追加: {pdf_file.name}")

print(f"\nZIP ファイルを作成しました: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
# AI Catalog にアップロード
CATALOG_NAME = "ec_market_survey_documents"

catalog_datasets = dr.Dataset.list()
existing = [d for d in catalog_datasets if d.name == CATALOG_NAME]

if existing:
    doc_dataset = existing[0]
    print(f"既存のカタログアイテムを使用します: {doc_dataset.name} (ID: {doc_dataset.id})")
else:
    doc_dataset = dr.Dataset.create_from_file(file_path=str(ZIP_PATH))
    print(f"カタログにアップロードしました: {doc_dataset.name} (ID: {doc_dataset.id})")

print(f"\nDataset ID: {doc_dataset.id}")

## 5. VectorDatabase の作成

以下の設定で VectorDatabase を作成します:

| パラメータ | 値 |
|---|---|
| Embedding モデル | `jina-embedding-t-en-v1` |
| チャンキング方式 | RECURSIVE |
| チャンクサイズ | 256 |
| オーバーラップ率 | 25% |
| セパレータ | `["\n\n", "\n", " ", ""]` |

In [ ]:
# Use Case を取得
existing_use_case_id = os.environ.get("DATAROBOT_DEFAULT_USE_CASE", "").strip()
if existing_use_case_id:
    use_case = dr.UseCase.get(existing_use_case_id)
else:
    # 前のノートブックで作成した Use Case を名前で検索
    use_cases = dr.UseCase.list()
    matching = [uc for uc in use_cases if "小売EC需要予測" in uc.name]
    if matching:
        use_case = matching[0]
    else:
        use_case = dr.UseCase.create(name="小売EC需要予測エージェント")

print(f"Use Case: {use_case.name} (ID: {use_case.id})")

In [ ]:
# VectorDatabase 作成 (REST API 経由)
# 利用可能なモデル:
#   intfloat/e5-large-v2, intfloat/e5-base-v2, intfloat/multilingual-e5-base,
#   intfloat/multilingual-e5-small, sentence-transformers/all-MiniLM-L6-v2,
#   jinaai/jina-embedding-t-en-v1, jinaai/jina-embedding-s-en-v2,
#   cl-nagoya/sup-simcse-ja-base
# 日本語PDFには cl-nagoya/sup-simcse-ja-base が最適

vdb_payload = {
    "name": "EC市場調査_VectorDatabase",
    "useCaseId": use_case.id,
    "datasetId": doc_dataset.id,
    "chunkingParameters": {
        "embeddingModel": "cl-nagoya/sup-simcse-ja-base",
        "chunkingMethod": "recursive",
        "chunkSize": 512,
        "chunkOverlapPercentage": 25,
        "separators": ["\n\n", "\n", " ", ""],
    },
}

print(f"VDB 作成リクエスト送信中...")
print(f"  Embedding: cl-nagoya/sup-simcse-ja-base (日本語特化)")
print(f"  Chunking: recursive, size=512, overlap=25%")

resp = requests.post(
    f"{API_BASE}/genai/vectorDatabases/",
    headers=HEADERS,
    json=vdb_payload,
)

if resp.status_code >= 400:
    print(f"\nエラー: {resp.status_code}")
    print(f"レスポンス: {resp.text}")
    resp.raise_for_status()

vdb_data = resp.json()
VDB_ID = vdb_data["id"]

print(f"\nVectorDatabase を作成しました。")
print(f"  VDB ID: {VDB_ID}")
print(f"  Status: {vdb_data.get('executionStatus', 'N/A')}")

## 6. VDB 完了の待機

VectorDatabase のインデックス構築が完了するまでポーリングします。

In [ ]:
def wait_for_vdb(vdb_id, timeout=600, interval=15):
    """VDB の構築完了をポーリングで待機する。"""
    start = time.time()
    while time.time() - start < timeout:
        resp = requests.get(
            f"{API_BASE}/genai/vectorDatabases/{vdb_id}/",
            headers=HEADERS,
        )
        resp.raise_for_status()
        data = resp.json()
        status = data.get("executionStatus", "UNKNOWN")
        print(f"  Status: {status} (elapsed: {time.time() - start:.0f}s)")

        if status == "COMPLETED":
            print("VDB 構築が完了しました。")
            return data
        elif status in ("ERROR", "FAILED"):
            raise RuntimeError(f"VDB 構築に失敗しました: {data}")

        time.sleep(interval)

    raise TimeoutError(f"VDB 構築が {timeout} 秒以内に完了しませんでした。")


print("VDB の構築完了を待機しています...")
vdb_result = wait_for_vdb(VDB_ID)
print(f"\nチャンク数: {vdb_result.get('size', 'N/A')}")

## 7-10. VDB → カスタムモデル → 登録 → デプロイ

VDB を検索可能なデプロイメントとして公開します。

In [ ]:
# ===== Step 1: VDB からカスタムモデルを作成 =====
print("VDB からカスタムモデルを作成中...")

# 方法1: toCustomModel エンドポイント
endpoints_to_try = [
    ("POST", f"{API_BASE}/genai/vectorDatabases/{VDB_ID}/toCustomModel/", {}),
    ("POST", f"{API_BASE}/genai/vectorDatabases/{VDB_ID}/customModel/", {}),
    ("POST", f"{API_BASE}/customModels/", {"vectorDatabaseId": VDB_ID, "name": "EC市場調査_VDB", "targetType": "TextGeneration", "targetName": "response"}),
]

custom_model_data = None
for method, url, payload in endpoints_to_try:
    print(f"  試行: {method} {url.replace(API_BASE, '...')}")
    try:
        if method == "POST":
            resp = requests.post(url, headers=HEADERS, json=payload)
        else:
            resp = requests.get(url, headers=HEADERS)
        
        if resp.status_code < 400:
            custom_model_data = resp.json()
            print(f"  成功! レスポンス keys: {list(custom_model_data.keys())}")
            break
        else:
            print(f"  失敗: {resp.status_code} - {resp.text[:200]}")
    except Exception as e:
        print(f"  例外: {e}")

if custom_model_data is None:
    # 方法2: VDB の詳細情報からカスタムモデル情報を取得
    print("\n直接 API で取得できなかったため、VDB 詳細から情報取得を試みます...")
    vdb_detail = requests.get(
        f"{API_BASE}/genai/vectorDatabases/{VDB_ID}/",
        headers=HEADERS,
    ).json()
    print(f"  VDB 詳細 keys: {list(vdb_detail.keys())}")
    print(f"  VDB 詳細 (抜粋): {json.dumps({k:v for k,v in vdb_detail.items() if 'model' in k.lower() or 'custom' in k.lower() or 'deploy' in k.lower() or 'status' in k.lower()}, indent=2, ensure_ascii=False)}")
    custom_model_data = vdb_detail

# カスタムモデル情報の抽出
CUSTOM_MODEL_VERSION_ID = custom_model_data.get("customModelVersionId") or custom_model_data.get("custom_model_version_id")
CUSTOM_MODEL_ID = custom_model_data.get("customModelId") or custom_model_data.get("custom_model_id") or custom_model_data.get("id")

print(f"\nCustom Model ID: {CUSTOM_MODEL_ID}")
print(f"Custom Model Version ID: {CUSTOM_MODEL_VERSION_ID}")

## ビルド待機 → 登録 → デプロイ

In [ ]:
# ===== Step 2: カスタムモデルビルドの待機 =====
if CUSTOM_MODEL_VERSION_ID and CUSTOM_MODEL_ID:
    print("カスタムモデルのビルド完了を待機しています...")
    start = time.time()
    for _ in range(40):  # 最大10分
        try:
            resp = requests.get(
                f"{API_BASE}/customModels/{CUSTOM_MODEL_ID}/versions/{CUSTOM_MODEL_VERSION_ID}/",
                headers=HEADERS,
            )
            if resp.status_code == 200:
                data = resp.json()
                build_status = data.get("buildStatus", "UNKNOWN")
                print(f"  Build Status: {build_status} (elapsed: {time.time() - start:.0f}s)")
                if build_status == "success":
                    print("ビルドが完了しました。")
                    break
                elif build_status in ("failed", "error"):
                    print(f"ビルドエラー: {data}")
                    break
            else:
                print(f"  ステータス確認失敗: {resp.status_code}")
        except Exception as e:
            print(f"  例外: {e}")
        time.sleep(15)
else:
    print("CUSTOM_MODEL_VERSION_ID が取得できていません。")
    print("VDB の詳細情報を再確認します...")
    vdb_detail = requests.get(
        f"{API_BASE}/genai/vectorDatabases/{VDB_ID}/",
        headers=HEADERS,
    ).json()
    print(json.dumps(vdb_detail, indent=2, ensure_ascii=False)[:2000])

## 登録とデプロイ

In [ ]:
# ===== Step 3: 登録 → デプロイ =====

if CUSTOM_MODEL_VERSION_ID:
    # モデルパッケージとして登録
    print("モデルパッケージとして登録中...")
    resp = requests.post(
        f"{API_BASE}/registeredModels/fromCustomModelVersion/",
        headers=HEADERS,
        json={
            "customModelVersionId": CUSTOM_MODEL_VERSION_ID,
            "registeredModelName": "EC市場調査_VectorDatabase",
        },
    )
    if resp.status_code >= 400:
        print(f"登録エラー: {resp.status_code} - {resp.text[:300]}")
        resp.raise_for_status()
    registered_data = resp.json()
    REGISTERED_MODEL_VERSION_ID = (
        registered_data.get("id")
        or registered_data.get("registeredModelVersionId")
    )
    print(f"  Registered Model Version ID: {REGISTERED_MODEL_VERSION_ID}")

    # デプロイ
    print("\nデプロイ中...")
    prediction_environment = dr.PredictionEnvironment.list()[0]
    print(f"  予測環境: {prediction_environment.name}")

    deployment = dr.Deployment.create(
        model_package_id=REGISTERED_MODEL_VERSION_ID,
        label="EC市場調査_VDB_デプロイメント",
        default_prediction_server_id=prediction_environment.id,
        use_case_id=use_case.id,
    )
    VDB_DEPLOYMENT_ID = deployment.id
    print(f"\n✅ デプロイ完了!")
    print(f"  VDB Deployment ID: {VDB_DEPLOYMENT_ID}")
else:
    print("⚠️ CUSTOM_MODEL_VERSION_ID が取得できなかったため、手動デプロイが必要です。")
    print(f"  VDB ID: {VDB_ID}")
    print("  DataRobot UI > GenAI > VectorDatabases から手動でデプロイしてください。")
    VDB_DEPLOYMENT_ID = "MANUAL_DEPLOY_REQUIRED"

## 結果出力

In [ ]:
print("=" * 60)
print("以下を .env ファイルに追記してください:")
print("=" * 60)
print(f"VDB_DEPLOYMENT_ID={VDB_DEPLOYMENT_ID}")
print("=" * 60)
print(f"\nVectorDatabase ID: {VDB_ID}")
print(f"Use Case ID:       {use_case.id}")
print("\n完了: 次のステップは 04_setup_prompt_and_mcp.ipynb に進んでください。")